In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, Input, backend as K
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# 1. PREPARE COMMON DATASET
# ==========================================
print("Loading CIFAR-10 and generating captions...")
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Normalize
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Class names
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

# Generate Captions
def generate_captions(labels):
    return np.array([f"A photo of a {class_names[l[0]]}" for l in labels])

train_texts_raw = generate_captions(y_train)
test_texts_raw = generate_captions(y_test)

# Vectorize Text
max_tokens = 50
seq_len = 10
vectorizer = layers.TextVectorization(max_tokens=max_tokens, output_sequence_length=seq_len)
vectorizer.adapt(train_texts_raw)

x_train_text = vectorizer(train_texts_raw).numpy()
x_test_text = vectorizer(test_texts_raw).numpy()

# Create Training Pairs (50% Positive, 50% Negative) for BOTH models
def create_pairs(images, texts, labels, num_pairs=5000):
    pair_imgs, pair_texts, pair_labels = [], [], []
    for i in range(num_pairs):
        if i % 2 == 0: # Positive
            pair_imgs.append(images[i])
            pair_texts.append(texts[i])
            pair_labels.append(1.0) # Match
        else: # Negative
            rand_idx = np.random.randint(0, len(texts))
            while labels[rand_idx] == labels[i]: rand_idx = np.random.randint(0, len(texts))
            pair_imgs.append(images[i])
            pair_texts.append(texts[rand_idx])
            pair_labels.append(0.0) # No Match
    return np.array(pair_imgs), np.array(pair_texts), np.array(pair_labels)

# Generate 5000 training pairs
print(" Creating training pairs...")
train_img, train_txt, train_lbl = create_pairs(x_train, x_train_text, y_train, num_pairs=5000)

# ==========================================
# 2. MODEL A: JOINT REPRESENTATION (FUSION)
# ==========================================
# Architecture: Concatenate -> Dense -> Sigmoid (0-1 probability)
i_in = Input(shape=(32,32,3))
t_in = Input(shape=(seq_len,))

# Image Branch
x = layers.Conv2D(16, (3,3), activation='relu')(i_in)
x = layers.GlobalAveragePooling2D()(x)
# Text Branch
y = layers.Embedding(max_tokens, 8)(t_in)
y = layers.GlobalAveragePooling1D()(y)

# FUSION
concat = layers.concatenate([x, y])
out = layers.Dense(1, activation='sigmoid')(concat) # Output probability of match

joint_model = models.Model(inputs=[i_in, t_in], outputs=out)
joint_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("\nTraining Joint Model...")
joint_model.fit([train_img, train_txt], train_lbl, epochs=5, batch_size=32, verbose=0)
print("Joint Model Trained.")

# ==========================================
# 3. MODEL B: COORDINATED REPRESENTATION (ALIGNMENT)
# ==========================================
# Architecture: Image Vec, Text Vec -> Cosine Similarity (Dot Product)
# Image Branch
i_enc = layers.Conv2D(16, (3,3), activation='relu')(i_in)
i_vec = layers.GlobalAveragePooling2D()(i_enc)
i_vec = layers.Dense(16)(i_vec) # Project to 16-dim
i_norm = layers.Lambda(lambda x: K.l2_normalize(x, axis=1))(i_vec)

# Text Branch
t_enc = layers.Embedding(max_tokens, 8)(t_in)
t_vec = layers.GlobalAveragePooling1D()(t_enc)
t_vec = layers.Dense(16)(t_vec) # Project to 16-dim
t_norm = layers.Lambda(lambda x: K.l2_normalize(x, axis=1))(t_vec)

# Dot Product (Cosine Similarity)
dot = layers.Dot(axes=1)([i_norm, t_norm])

coord_model = models.Model(inputs=[i_in, t_in], outputs=dot)
# For Cosine, 1.0 is match, 0.0 (or -1) is mismatch. We use MSE to regress to the label (1 or 0).
coord_model.compile(optimizer='adam', loss='mse')

print("\nTraining Coordinated Model...")
coord_model.fit([train_img, train_txt], train_lbl, epochs=5, batch_size=32, verbose=0)
print("Coordinated Model Trained.")

# ==========================================
# 4. THE HEAD-TO-HEAD BATTLE
# ==========================================
print("\n STARTING BATTLE: 100 Rounds of 'Pick the Correct Caption'")

joint_score = 0
coord_score = 0
rounds = 100

for i in range(rounds):
    # 1. Setup the Challenge: 1 Image, 5 Captions (1 Right, 4 Wrong)
    curr_img = x_test[i]
    true_idx = i

    # Get 4 random wrong indices
    wrong_indices = []
    while len(wrong_indices) < 4:
        r = np.random.randint(0, len(x_test))
        if y_test[r] != y_test[true_idx]: # Ensure it's actually different class
            wrong_indices.append(r)

    # Prepare batch for prediction (Shape: 5 images, 5 texts)
    # We repeat the image 5 times to pair with the 5 texts
    batch_imgs = np.array([curr_img] * 5)

    candidate_indices = [true_idx] + wrong_indices
    batch_texts = x_test_text[candidate_indices] # The 5 text vectors

    # 2. ASK JOINT MODEL
    # It outputs probability (0-1). We want the highest probability.
    preds_j = joint_model.predict([batch_imgs, batch_texts], verbose=0)
    best_j = np.argmax(preds_j) # Index of highest score

    # 3. ASK COORDINATED MODEL
    # It outputs similarity. We want highest similarity.
    preds_c = coord_model.predict([batch_imgs, batch_texts], verbose=0)
    best_c = np.argmax(preds_c) # Index of highest score

    # 4. SCORING
    # The correct answer is always at index 0 (because we put true_idx first in the list)
    if best_j == 0: joint_score += 1
    if best_c == 0: coord_score += 1

# ==========================================
# 5. FINAL RESULTS
# ==========================================
print("\n========================================")
print(f"FINAL SCORE (Accuracy on {rounds} test images)")
print("========================================")
print(f"Joint Model Accuracy:       {joint_score}%")
print(f"Coordinated Model Accuracy: {coord_score}%")
print("========================================")

if joint_score > coord_score:
    print("WINNER: Joint Representation (Deep Fusion)")

elif coord_score > joint_score:
    print("WINNER: Coordinated Representation (Metric Learning)")

else:
    print("DRAW")